# Multi-Agent Payment Orchestrator

## 개요

이 튜토리얼에서는 agent별 budget, multi-wallet 지원, 전체 지출 attribution을 갖춘 multi-agent system을 구축한 다음 budget이 소진되었을 때의 지능형 failover를 보여 줍니다.

### 이 튜토리얼에서 확인할 내용

단순 병렬 실행을 넘어 orchestration이 필요한 세 가지 pattern을 살펴봅니다.

| Demo | Pattern | 검증 내용 |
|------|---------|---------------|
| **Demo 1** | Spend Attribution | 두 wallet, 두 budget, agent별 전체 비용 추적 |
| **Demo 2** | Budget Exhaustion + Failover | Orchestrator가 payment 거부를 감지하고 정상 agent로 rerouting |
| **Demo 3** | Structural Safety | Orchestrator는 `http_request`가 있어도 지출 불가 |

### 핵심 개념

| 개념 | 작동 방식 |
|---------|-------------|
| **Multi-wallet** | 하나의 PaymentManager, 두 Connector(Coinbase + Privy), 두 Instrument |
| **Agent별 budget** | 독립적인 지출 limit이 있는 별도 Session |
| **Orchestrator 분리** | Orchestrator에는 plugin이 없어 지출할 수 없으며 routing만 수행 |
| **Budget 소진 + failover** | 한 agent의 budget이 거부되면 orchestrator가 다른 agent로 rerouting |
| **Role 적용** | Runtime의 ProcessPaymentRole을 통해 deterministic code가 payment를 처리하고 agent는 session 생성 불가 |

> **Testnet 전용입니다.** 모든 코드는 [faucet.circle.com](https://faucet.circle.com/)의 무료 USDC와 함께 Base Sepolia를 사용합니다. Testnet USDC에는 실제 가치가 없습니다.

### 아키텍처

![Architecture Overview](images/architecture_overview.png)


### Payment Flow — 세 가지 Demo

![Payment Flow — Three Demos](images/payment_flow_three_demos.png)


### 튜토리얼 세부 정보

| 항목                | 세부 정보                                                            |
|:--------------------|:---------------------------------------------------------------------|
| Tutorial type       | Task 기반                                                            |
| Agent type          | Multi-agent(orchestrator + specialist 2개)                           |
| Agentic Framework   | Strands Agents(agents-as-tools pattern)                              |
| LLM model           | Anthropic Claude Sonnet                                              |
| Tutorial 구성 요소  | AgentCore payments(multi-session), AgentCore Runtime, AgentCore CLI  |
| 예제 난이도         | 고급                                                                 |
| 사용 SDK            | bedrock-agentcore SDK, Strands Agents SDK, AgentCore CLI             |

## 사전 요구 사항

* Tutorial 00b 완료(Coinbase와 Privy가 모두 있는 multi-provider `.env`)
* 두 wallet 모두 https://faucet.circle.com/의 testnet USDC 입금 완료
* AgentCore CLI: `npm install -g @aws/agentcore`
* Docker 설치(배포 중 container build에 사용)
* `pip install -r requirements.txt`

In [ ]:
%pip install -r requirements.txt --quiet

## 1단계 — AWS Credentials 검증

In [ ]:
import os
import sys
import json

sys.path.append("..")

import boto3
from dotenv import load_dotenv

load_dotenv(override=True)

# 이름이 지정된 AWS profile을 사용하려면 주석 해제:
# os.environ['AWS_PROFILE'] = '<your-profile>'

session = boto3.Session()
identity = session.client("sts").get_caller_identity()
print(f"✅ Authenticated as: {identity['Arn']}")
print(f"   Region: {session.region_name}")

## 2단계 — Multi-Provider Config 불러오기

In [ ]:
from utils import load_tutorial_env, print_summary

config = load_tutorial_env()

if not config.get("multi_provider"):
    raise ValueError("This tutorial requires a multi-provider config. Run 00b_multi_provider_setup.ipynb first.")

PAYMENT_MANAGER_ARN = config["payment_manager_arn"]
REGION = config["region"]
USER_ID = config["user_id"]

COINBASE = config["instruments"]["coinbase"]
PRIVY = config["instruments"]["stripe_privy"]

MODEL_ID = os.environ.get("MODEL_ID", "us.anthropic.claude-sonnet-4-6")

print_summary(
    "Multi-Provider Config",
    manager_arn=PAYMENT_MANAGER_ARN,
    coinbase_instrument=COINBASE["instrument_id"],
    privy_instrument=PRIVY["instrument_id"],
)

## 3단계 — Instrument 검증 및 Agent별 Session 생성

App backend(ManagementRole)는 두 instrument가 ACTIVE이고 자금이 입금되었는지 검증한 다음 독립적인 budget이 있는 두 session을 생성합니다.
- Session A: Research Agent용 $0.50(Coinbase wallet)
- Session B: Discovery Agent용 $0.20(Privy wallet)

총 할당액은 $0.70이며 agent는 서로의 지출에 영향을 줄 수 없습니다.

In [ ]:
from bedrock_agentcore.payments import PaymentManager

# Tutorial 00b의 기존 Payment Manager ARN을 래핑하는 SDK client
manager = PaymentManager(payment_manager_arn=PAYMENT_MANAGER_ARN, region_name=REGION)

# 계속 진행하기 전에 두 instrument가 모두 ACTIVE이고 자금이 입금되었는지 검증
for label, instr_id in [
    ("Coinbase", COINBASE["instrument_id"]),
    ("Privy", PRIVY["instrument_id"]),
]:
    instr = manager.get_payment_instrument(user_id=USER_ID, payment_instrument_id=instr_id)
    status = instr.get("status", "UNKNOWN")
    assert status == "ACTIVE", f"{label} instrument {instr_id} is {status} — fund and delegate first"
    print(f"✅ {label} instrument {instr_id} is {status}")

# Session A: Research Agent — 더 큰 budget, Coinbase wallet
sess_a = manager.create_payment_session(
    user_id=USER_ID,
    limits={"maxSpendAmount": {"value": "0.50", "currency": "USD"}},
    expiry_time_in_minutes=60,
)
SESSION_A_ID = sess_a["paymentSessionId"]

# Session B: Discovery Agent — 더 작은 budget, Privy wallet
sess_b = manager.create_payment_session(
    user_id=USER_ID,
    limits={"maxSpendAmount": {"value": "0.20", "currency": "USD"}},
    expiry_time_in_minutes=60,
)
SESSION_B_ID = sess_b["paymentSessionId"]

print_summary(
    "Per-Agent Sessions",
    session_a=f"{SESSION_A_ID} ($0.50, Coinbase)",
    session_b=f"{SESSION_B_ID} ($0.20, Privy)",
    total_allocated="$0.70",
)

---

# Part A: 로컬 실행

먼저 multi-agent system을 로컬에서 실행하여 빠르게 결과를 확인하고 개념을 익힙니다.

## 4단계 — Plugin 및 Agent 생성

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands.tools import tool
from strands_tools import http_request
from bedrock_agentcore.payments.integrations.strands import (
    AgentCorePaymentsPlugin,
    AgentCorePaymentsPluginConfig,
)

# --- Plugin(agent별 하나, 서로 다른 session + instrument) ---

research_plugin = AgentCorePaymentsPlugin(
    config=AgentCorePaymentsPluginConfig(
        payment_manager_arn=PAYMENT_MANAGER_ARN,
        user_id=USER_ID,
        payment_instrument_id=COINBASE["instrument_id"],
        payment_session_id=SESSION_A_ID,
        region=REGION,
        network_preferences_config=["eip155:84532", "base-sepolia"],
    )
)

discovery_plugin = AgentCorePaymentsPlugin(
    config=AgentCorePaymentsPluginConfig(
        payment_manager_arn=PAYMENT_MANAGER_ARN,
        user_id=USER_ID,
        payment_instrument_id=PRIVY["instrument_id"],
        payment_session_id=SESSION_B_ID,
        region=REGION,
        network_preferences_config=["eip155:84532", "base-sepolia"],
    )
)

# --- Budget 확인 tool(orchestrator 전용) ---


@tool
def check_budgets() -> str:
    """Check remaining budget for each specialist agent.

    Returns:
        JSON with per-agent spend and remaining budget.
    """
    results = {}
    for label, sid in [
        ("research_agent", SESSION_A_ID),
        ("discovery_agent", SESSION_B_ID),
    ]:
        info = manager.get_payment_session(
            user_id=USER_ID,
            payment_session_id=sid,
        )
        sess = info
        results[label] = {
            "session_id": sid,
            "available": sess.get("availableLimits", {}).get("availableSpendAmount", "N/A"),
            "budget": sess.get("limits", {}).get("maxSpendAmount", "N/A"),
        }
    return json.dumps(results, indent=2)


# --- 전문 agent ---

model = BedrockModel(model_id=MODEL_ID, streaming=True)

research_agent = Agent(
    model=model,
    tools=[http_request],
    plugins=[research_plugin],
    system_prompt=(
        "You are a research specialist. Use http_request to access paid endpoints "
        "on the Coinbase Bazaar (Base Sepolia testnet). "
        "IMPORTANT: Only use GET requests. Never use POST, PUT, or DELETE. "
        'When you discover endpoints from the Bazaar, look for the URL in the "resource" field of the response. '
        "Payment is handled automatically via x402. "
        "Report what data you found and what it cost."
    ),
)

discovery_agent = Agent(
    model=model,
    tools=[http_request],
    plugins=[discovery_plugin],
    system_prompt=(
        "You are a data discovery specialist. Use http_request to access paid "
        "endpoints on the Coinbase Bazaar (Base Sepolia testnet). "
        "IMPORTANT: Only use GET requests. Never use POST, PUT, or DELETE. "
        'When you discover endpoints from the Bazaar, look for the URL in the "resource" field of the response. '
        "Payment is handled automatically via x402. Report what you found and the cost."
    ),
)

# --- Orchestrator(plugin 없음) ---

orchestrator = Agent(
    model=model,
    tools=[
        research_agent.as_tool(
            name="research_agent",
            description="Research specialist with Coinbase wallet and $0.50 budget. Use for paid data lookups.",
        ),
        discovery_agent.as_tool(
            name="discovery_agent",
            description="Discovery specialist with Privy wallet and $0.20 budget. Use for cheap paid endpoints.",
        ),
        check_budgets,
    ],
    system_prompt=(
        "You are an orchestrator that coordinates specialist agents.\n"
        "- research_agent: paid data lookups (budget: $0.50, Coinbase wallet)\n"
        "- discovery_agent: cheap paid endpoints (budget: $0.20, Privy wallet)\n"
        "- check_budgets: monitor spend across both agents\n\n"
        "You cannot make payments yourself. Only the specialists can spend.\n"
        "If one agent's budget is exhausted, route remaining work to the other.\n"
        "After tasks complete, check budgets and report total spend."
    ),
)

print("\u2705 Research Agent: Session A + Coinbase + $0.50 budget")
print("\u2705 Discovery Agent: Session B + Privy + $0.20 budget")
print("\u2705 Orchestrator: NO plugin (cannot spend)")

---

## 5단계 — 세 가지 Multi-Agent Demo 실행

각 demo는 단순 병렬 실행을 넘어 orchestration이 필요한 고유 기능을 보여 줍니다. 실행 후에는 정확한 결과를 검증하는 spend report가 이어집니다.

| Demo | Pattern | 확인할 내용 |
|------|---------|-------------------|
| 1 | **Spend Attribution** | 두 wallet에서 독립적으로 차감되고 agent별로 전체 추적 |
| 2 | **Budget Exhaustion + Failover** | Research agent 거부 → orchestrator rerouting → discovery agent 성공 |
| 3 | **Structural Safety** | Orchestrator가 402를 받고 결제할 수 없어 spend report가 변경되지 않음 |

## Demo 1 — Spend Attribution

두 agent가 서로 다른 유료 endpoint를 호출합니다. 실행 후 어떤 wallet에서 무엇을 결제했는지 정확히 확인합니다. 두 wallet, 두 budget, 전체 attribution의 기준이 되는 demo입니다.

- Research Agent(Coinbase, $0.50) → Bazaar에서 `weather`를 검색하고 endpoint 호출
- Discovery Agent(Privy, $0.20) → Bazaar에서 `market news`를 검색하고 endpoint 호출

In [ ]:
result = orchestrator(
    "I need two things:\n"
    "1. Ask the research_agent to search the Bazaar for weather endpoints on Base Sepolia "
    "(GET https://api.cdp.coinbase.com/platform/v2/x402/discovery/search?query=weather&network=base-sepolia&limit=3). "
    'From the results, find the endpoint URL in the "resource" field and call it with http_request using GET. '
    "Report the weather data and cost.\n"
    "2. Ask the discovery_agent to search the Bazaar for market+news endpoints on Base Sepolia "
    "(GET https://api.cdp.coinbase.com/platform/v2/x402/discovery/search?query=market+news&network=base-sepolia&limit=3). "
    'From the results, find the endpoint URL in the "resource" field and call it with http_request using GET '
    "(include query params: batchSize=24, targetBlocks=3, urgency=balanced). "
    "Report the payout plan data and cost.\n\n"
    "After both tasks complete, check the budgets and give me a spend report showing what each agent spent from their respective wallets."
)
print(result.message)

### 📊 Spend Report — Demo 1

각 session은 지출을 독립적으로 추적합니다. `available` 및 `budget` field를 비교하면 각 agent가 해당 wallet에서 지출한 금액을 확인할 수 있습니다.

In [ ]:
for label, sid, wallet_provider in [
    ("Research Agent (Coinbase)", SESSION_A_ID, "Coinbase"),
    ("Discovery Agent (Privy)", SESSION_B_ID, "Privy"),
]:
    info = manager.get_payment_session(
        user_id=USER_ID,
        payment_session_id=sid,
    )
    sess = info
    print_summary(
        label,
        session_id=sid,
        wallet=wallet_provider,
        available=sess.get("availableLimits", {}).get("availableSpendAmount", "N/A"),
        budget=sess.get("limits", {}).get("maxSpendAmount", "N/A"),
    )

## Demo 2 — Budget 소진 및 Failover

이 부분이 multi-agent의 핵심 효과입니다. Research agent에 유료 호출 비용을 감당할 수 없을 만큼 **작은** budget을 제공합니다. Orchestrator는 다음 작업을 수행해야 합니다.

1. 먼저 task를 research_agent로 routing
2. Payment 실패 관찰(budget 초과로 AgentCore가 API level에서 거부)
3. `check_budgets`를 호출하여 확인
4. Budget이 남아 있는 discovery_agent로 rerouting
5. 발생한 결과 보고

이는 독립적인 두 agent만으로는 수행할 수 없습니다. Orchestrator가 상황을 관찰하고 적응하여 복구합니다.

In [ ]:
# 유료 x402 호출에 부족한 $0.0005의 tiny session 생성
tiny_sess = manager.create_payment_session(
    user_id=USER_ID,
    limits={"maxSpendAmount": {"value": "0.0005", "currency": "USD"}},
    expiry_time_in_minutes=60,
)
TINY_SESSION_ID = tiny_sess["paymentSessionId"]
print(f"Tiny research session: {TINY_SESSION_ID} (budget: $0.0005)")
print(f"Discovery session: {SESSION_B_ID} (budget: $0.20)")

# Tiny session으로 research plugin 다시 구축
tiny_research_plugin = AgentCorePaymentsPlugin(
    config=AgentCorePaymentsPluginConfig(
        payment_manager_arn=PAYMENT_MANAGER_ARN,
        user_id=USER_ID,
        payment_instrument_id=COINBASE["instrument_id"],
        payment_session_id=TINY_SESSION_ID,
        region=REGION,
        network_preferences_config=["eip155:84532", "base-sepolia"],
    )
)

tiny_research_agent = Agent(
    model=model,
    tools=[http_request],
    plugins=[tiny_research_plugin],
    system_prompt=(
        "You are a research specialist. Use http_request to access paid endpoints "
        "on the Coinbase Bazaar (Base Sepolia testnet). "
        "IMPORTANT: Only use GET requests. Never use POST, PUT, or DELETE. "
        "Payment is handled automatically via x402. "
        "Report what data you found and what it cost. If payment fails, report the failure clearly."
    ),
)


# Budget 실패로 orchestrator가 중단되지 않도록 research agent에 error handling 적용
@tool
def research_agent_tool(task: str) -> str:
    """Research specialist with VERY SMALL budget ($0.0005, Coinbase wallet). Likely to fail on paid calls due to budget exhaustion."""
    try:
        result = tiny_research_agent(task)
        return result.message.get("content", [{}])[0].get("text", str(result))
    except Exception as e:
        return f"PAYMENT FAILED — budget exhausted. Error: {str(e)}"


@tool
def check_budgets_v2() -> str:
    """Check remaining budget for the research and discovery agents. Use this after a payment failure to confirm budget exhaustion."""
    results = {}
    for label, sid in [
        ("research_agent", TINY_SESSION_ID),
        ("discovery_agent", SESSION_B_ID),
    ]:
        info = manager.get_payment_session(user_id=USER_ID, payment_session_id=sid)
        results[label] = {
            "session_id": sid,
            "available": info.get("availableLimits", {}).get("availableSpendAmount", "N/A"),
            "budget": info.get("limits", {}).get("maxSpendAmount", "N/A"),
        }
    return json.dumps(results, indent=2)


failover_orchestrator = Agent(
    model=model,
    tools=[
        research_agent_tool,
        discovery_agent.as_tool(
            name="discovery_agent",
            description="Discovery specialist with healthy budget ($0.20) and Privy wallet. Use as fallback when research_agent fails.",
        ),
        check_budgets_v2,
    ],
    system_prompt=(
        "You are an orchestrator that coordinates specialist agents.\n"
        "- research_agent_tool: research specialist, budget $0.0005 (extremely tight!, Coinbase wallet)\n"
        "- discovery_agent: budget $0.20 (healthy, Privy wallet)\n"
        "- check_budgets_v2: monitor spend across both agents\n\n"
        "You cannot make payments yourself. Only the specialists can spend.\n"
        "IMPORTANT: If research_agent_tool fails due to budget exhaustion, call check_budgets_v2 to confirm, "
        "then route the work to discovery_agent as a fallback.\n"
        "Report what happened — which agent succeeded, which failed, and why."
    ),
)

print("\n✅ Failover orchestrator ready")
print("   research_agent_tool: $0.0005 budget (will be rejected)")
print("   discovery_agent: $0.20 budget (healthy fallback)")
print("   check_budgets_v2: monitors both sessions")

In [ ]:
result = failover_orchestrator(
    "I need a fortune reading. "
    "Use research_agent_tool to call GET https://x402-test.genesisblock.ai/api/market-news — "
    "this is a paid x402 endpoint ($0.002).\n\n"
    "If research_agent_tool fails (budget exceeded), call check_budgets_v2 to confirm, "
    "then ask the discovery_agent to call the same endpoint instead.\n"
    "Report which agent completed the task and why the other failed."
)
print(result.message)

### 📊 Spend Report — Demo 2

$0.0005의 session budget으로 $0.002 호출을 처리할 수 없어 AgentCore가 research agent의 payment를 API level에서 거부했습니다. Orchestrator가 실패를 감지하고 discovery agent로 rerouting했습니다.

예상 결과: Research Agent budget은 변경되지 않고(payment 거부), Discovery Agent budget은 감소합니다(payment 성공).

In [ ]:
for label, sid, wallet_provider in [
    ("Research Agent — EXHAUSTED (Coinbase)", TINY_SESSION_ID, "Coinbase"),
    ("Discovery Agent — FALLBACK (Privy)", SESSION_B_ID, "Privy"),
]:
    info = manager.get_payment_session(user_id=USER_ID, payment_session_id=sid)
    sess = info
    print_summary(
        label,
        session_id=sid,
        wallet=wallet_provider,
        available=sess.get("availableLimits", {}).get("availableSpendAmount", "N/A"),
        budget=sess.get("limits", {}).get("maxSpendAmount", "N/A"),
    )

## Demo 3 — Structural Safety(Orchestrator 지출 불가)

Orchestrator에는 **payment plugin이 없습니다**. LLM이 유료 endpoint를 직접 호출하기로 결정해도 402를 intercept하고 transaction에 sign할 plugin이 없으므로 x402 payment flow를 실행할 수 없습니다.

이는 prompt level이 아닌 구조적 적용입니다. Orchestrator의 role은 **payment**가 아니라 **routing**입니다.

In [ ]:
# Orchestrator에 http_request를 직접 제공하여 지출 가능 여부 확인
from strands_tools import http_request as http_tool

unsafe_orchestrator = Agent(
    model=model,
    tools=[http_tool],  # http_request는 있지만 payment plugin은 없음
    system_prompt=(
        "You have http_request available. Try to access this paid endpoint: "
        "GET https://x402-test.genesisblock.ai/api/weather"
        "Report exactly what happens."
    ),
)

result = unsafe_orchestrator(
    "Call GET https://x402-test.genesisblock.ai/api/weather and tell me what you get back. "
    "This is a paid x402 endpoint. Report the HTTP status and response."
)
print(result.message)

### 📊 Spend Report — Demo 3

Orchestrator가 endpoint의 결제 요구를 나타내는 **402 Payment Required** response를 받았습니다. 하지만 payment plugin이 없으므로 다음 작업을 수행할 수 없습니다.
1. Response에서 x402 payment requirement parsing
2. `ProcessPayment`를 호출하여 transaction에 sign
3. Payment proof header와 함께 retry

402에서 flow가 종료됩니다. 아래 spend report에서 **budget이 전혀 사용되지 않았음**을 확인할 수 있습니다.

| Agent | Plugin 보유 | 지출 가능 여부 |
|-------|-----------|-----------|
| Orchestrator | ❌ 없음 | ❌ `http_request`가 있어도 결제 불가 |
| Research Agent | ✅ 있음(Coinbase) | ✅ Session budget 내에서 가능 |
| Discovery Agent | ✅ 있음(Privy) | ✅ Session budget 내에서 가능 |

In [ ]:
# 확인: 지출이 발생하지 않음 — orchestrator에는 plugin이 없어 결제 불가
for label, sid, wallet_provider in [
    ("Research Agent (Coinbase)", SESSION_A_ID, "Coinbase"),
    ("Discovery Agent (Privy)", SESSION_B_ID, "Privy"),
]:
    info = manager.get_payment_session(user_id=USER_ID, payment_session_id=sid)
    sess = info
    print_summary(
        f"{label} — NO CHANGE",
        session_id=sid,
        wallet=wallet_provider,
        available=sess.get("availableLimits", {}).get("availableSpendAmount", "N/A"),
        budget=sess.get("limits", {}).get("maxSpendAmount", "N/A"),
    )

print("↑ Budgets unchanged — the orchestrator's 402 attempt spent nothing.")

---

# Part B: AgentCore Runtime에 배포

동일한 multi-agent system을 ProcessPaymentRole 적용 및 AgentCore Observability와 함께 배포합니다.

## 7단계 — Agent 코드 검토

`payment_orchestrator.py` 파일은 세 agent를 하나의 `BedrockAgentCoreApp`으로 package합니다. Entrypoint는 invocation payload를 통해 app backend에서 두 session ID와 두 instrument ID를 받습니다.

In [ ]:
with open("payment_orchestrator.py") as f:
    print(f.read())

## 8단계 — AgentCore CLI로 배포

> **비용 안내:** AgentCore Runtime 및 online evaluation에는 invocation별 요금이 발생합니다. 작업을 마치면 리소스 정리 섹션을 참조하세요.

```bash
# 아직 생성하지 않았다면 프로젝트 생성
agentcore create --name PaymentOrchestrator --defaults

# Agent 추가
agentcore add agent \
  --name PaymentOrchestrator \
  --type byo \
  --code-location . \
  --entrypoint payment_orchestrator.py

# 배포(Observability는 자동으로 활성화됨)
agentcore deploy -y

# 배포와 Observability 확인
agentcore status
```

이제 두 observability 계층이 활성화됩니다.

| 계층 | 수집 항목 | 설정 주체 |
|-------|-----------------|----------|
| **Agent observability** | LLM 호출, tool 실행, agent trace | `agentcore deploy`(자동) |
| **Payment observability** | ProcessPayment trace, budget 차감, vended log | Tutorial 00의 `enable_observability()` |

두 계층 모두 AgentCore Observability(CloudWatch)로 전달됩니다. Service map에는 Orchestrator → Specialist → AgentCore payments의 전체 flow가 표시됩니다.

Runtime은 **ProcessPaymentRole**로 실행됩니다. 동일한 manager의 두 instrument에 대해 `ProcessPayment`를 호출할 수 있지만 session 생성 또는 budget 수정은 할 수 없습니다.

### 설정할 Environment Variable

```bash
PAYMENT_MANAGER_ARN=<your-manager-arn>
AWS_REGION=us-west-2
MODEL_ID=us.anthropic.claude-sonnet-4-6
```

## 8b단계 — Online Evaluation 추가

AgentCore Evaluations는 모든 invocation을 자동으로 채점하는 built-in evaluator를 제공합니다. Payment orchestrator와 가장 관련 있는 evaluator는 다음과 같습니다.

| Evaluator | Level | 확인 항목 |
|-----------|-------|---------------|
| `Builtin.GoalSuccessRate` | Session | Orchestrator가 research와 discovery task를 모두 완료했는가? |
| `Builtin.ToolSelectionAccuracy` | Tool | 각 task를 적절한 specialist로 routing했는가? |
| `Builtin.Helpfulness` | Trace | Spend report가 명확하고 유용한가? |

```bash
# 기본 제공 evaluator를 사용하는 online evaluation 추가 - invocation 100% 평가
agentcore add online-eval \
  --name PaymentMonitor \
  --runtime PaymentOrchestrator \
  --evaluator Builtin.GoalSuccessRate Builtin.ToolSelectionAccuracy Builtin.Helpfulness \
  --sampling-rate 100 \
  --enable-on-create

# 배포
agentcore deploy -y
```

배포 후 score가 CloudWatch의 AgentCore Observability dashboard에 표시됩니다. 결과를 확인합니다.

```bash
# Evaluation 기록 확인
agentcore evals history --runtime PaymentOrchestrator --limit 5

# Evaluation 로그 스트리밍
agentcore logs evals --runtime PaymentOrchestrator --since 1h
```

Custom evaluator 코드는 필요하지 않습니다. Built-in evaluator가 채점을 처리하며 결과는 payment trace와 동일한 CloudWatch dashboard로 전달됩니다.

## 9단계 — App Backend에서 호출

App backend가 새 session을 생성하고 두 session ID 및 instrument ID와 함께 배포된 orchestrator를 호출합니다.

In [ ]:
# App backend가 배포된 orchestrator에 보내는 invocation payload
invocation_payload = {
    "prompt": (
        "Search the Bazaar for weather endpoints on Base Sepolia "
        "(GET https://api.cdp.coinbase.com/platform/v2/x402/discovery/search?query=weather&network=base-sepolia&limit=3) "
        'and ask the research_agent to call the endpoint from the "resource" field. '
        "Then search for market news endpoints "
        "(GET https://api.cdp.coinbase.com/platform/v2/x402/discovery/search?query=market+news&network=base-sepolia&limit=3) "
        "and ask the discovery_agent to call that endpoint with batchSize=24&targetBlocks=3&urgency=balanced. "
        "Check budgets after and report total spend per agent."
    ),
    "user_id": USER_ID,
    "research_session_id": SESSION_A_ID,
    "research_instrument_id": COINBASE["instrument_id"],
    "discovery_session_id": SESSION_B_ID,
    "discovery_instrument_id": PRIVY["instrument_id"],
}

print("Invocation payload:")
print(json.dumps(invocation_payload, indent=2))
print("\nInvoke with:")
print(f"  agentcore invoke '{json.dumps(invocation_payload)}'")

## 10단계 — CloudWatch에서 Trace 보기

Observability를 활성화하면 두 agent의 모든 `ProcessPayment` 호출에서 trace가 생성됩니다. CloudWatch X-Ray에서 다음 내용을 확인할 수 있습니다.

- **별도의 두 payment flow** — Research Agent trace(Session A, Coinbase) 및 Discovery Agent trace(Session B, Privy)
- **Budget 차감** — 각 payment의 3단계 flow(reserve → sign → commit)
- **Budget 소진** — research agent가 $0.50 limit에 도달하면 실패한 ProcessPayment trace에 budget exceeded error 표시
- **Failover** — research agent 실패 직후 discovery agent의 성공한 payment

Vended log에는 session별 지출 진행 상황이 표시됩니다. Session B가 정상 상태를 유지하는 동안 Session A의 $0.50 budget이 감소하는 것을 확인할 수 있습니다.

```bash
# 최근 trace 목록 확인
agentcore traces list --limit 20

# 로그 스트리밍
agentcore logs
```

In [ ]:
print("CloudWatch X-Ray:")
print(f"  https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#xray:traces")
print("\nVended logs:")
print(f"  https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#logsV2:log-groups")
print("\nGenAI Dashboard:")
print(f"  https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#gen-ai-observability/agent-core")

## 요약

전체 budget governance를 갖춘 multi-agent payment system을 구축하고 AgentCore Runtime에 배포했습니다.

### 확인한 세 가지 Orchestration Pattern

| Demo | 실행 결과 | 중요한 이유 |
|------|--------------|----------------|
| **Spend Attribution** | 두 agent가 별도 wallet에서 지출하고 session별 추적을 통해 결제 주체와 항목 확인 | 공유 state 없이 전체 비용 할당 |
| **Budget Exhaustion + Failover** | Research agent의 payment가 작은 budget으로 인해 거부되고 orchestrator가 discovery agent로 rerouting | 독립적인 agent로는 구현할 수 없는 적응형 동작 |
| **Structural Safety** | Orchestrator가 유료 endpoint를 직접 호출하여 결제 수단 없이 402 수신 | Prompt가 아니라 아키텍처에서 지출 규칙 적용 |

### 구성 요소별 역할

| 구성 요소 | 수행 작업 | Role |
|-----|----------|------|
| App backend | Session 생성, budget 할당, orchestrator 호출 | ManagementRole |
| Orchestrator | Task routing, budget monitoring | NONE(plugin 없음) |
| Research Agent | 유료 endpoint 호출, Session A에서 지출 | ProcessPaymentRole(Coinbase) |
| Discovery Agent | 유료 endpoint 호출, Session B에서 지출 | ProcessPaymentRole(Privy) |
| CloudWatch | 모든 payment trace 및 budget 진행 상황 표시 | 자동 |
| Online eval | 각 session의 budget 준수 및 routing quality 채점 | Built-in evaluator |

### Resource 계층

```
PaymentManager
  ├── CoinbaseCDP Connector → Research Agent (Session A, $0.50)
  └── StripePrivy Connector → Discovery Agent (Session B, $0.20)
```

동일한 manager, 서로 다른 connector와 wallet, 독립적인 budget을 사용하며 CloudWatch에서 전체 attribution을 확인할 수 있습니다.

## 검증

위 셀이 오류 없이 실행되었다면 multi-agent payment orchestrator가 배포된 것입니다. 위의 세 demo 모두에서 spend report가 출력되고 각 agent의 session에 감소한 `availableLimit`이 표시되어 agent별 budget 적용이 올바르게 작동함을 확인할 수 있습니다.

## 리소스 정리(선택 사항)

> **리소스 정리 전:** `agentcore remove all`을 실행하면 deployment, CloudWatch log, evaluation result가 영구적으로 삭제됩니다. 먼저 보관할 data를 내보내세요.

Session은 `expiryTimeInMinutes`가 지나면 자동으로 만료됩니다.

**이 튜토리얼을 다시 실행할 계획이라면 리소스를 정리하지 마세요.** Payment resource가 재사용됩니다.

모든 튜토리얼을 마친 후에만 리소스를 정리하세요.

```bash
# 배포된 orchestrator 제거
# agentcore remove all -y

# 결제 리소스(Manager, Connector, Instrument): Tutorial 00b의 cleanup 실행
```

# 축하합니다!

Agent별 payment limit, multi-wallet 지원, Runtime의 role 적용, 전체 spend attribution을 갖춘 multi-agent payment orchestrator를 구축했습니다.

### 다음 단계

- **Custom payment evaluator** — `agentcore add evaluator`로 budget 준수, 지출 투명성, routing quality를 기준으로 session을 채점하는 LLM-as-a-Judge evaluator 생성
- **CloudWatch alarm** — payment 실패율 및 eval score 하락에 alarm을 설정하여 agent 지출 동작이 저하될 때 알림 수신
- **On-demand evaluation** — 새 prompt version을 배포하기 전에 과거 trace에 `agentcore run eval`을 실행하여 변경 사항 benchmark
- **Tenant별 budget** — 고객 또는 부서별로 budget을 분리하도록 multi-session pattern을 확장하고 CloudWatch에서 전체 attribution 확인
- **AgentCore Gateway로 확장** — Tutorial 04의 Gateway target을 AgentCore Runtime 및 payments와 함께 재사용. Bazaar를 직접 호출하는 대신 두 agent를 하나의 Gateway endpoint를 통해 routing하며 payment infrastructure는 동일하게 유지

---

## 선택 사항: AgentCore Gateway를 통한 Agent Routing

Tutorial 04를 완료하여 Coinbase Bazaar target이 있는 Gateway를 보유했다면 `http_request`를 MCP tool로 교체할 수 있습니다. Agent는 raw HTTP URL 대신 `search_resources`와 `proxy_tool_call`을 사용하므로 대규모 구축 방식에 더 가깝습니다.

Payment infrastructure는 동일하게 유지되고 tool 계층만 변경됩니다.

```python
from datetime import timedelta
from mcp.client.streamable_http import streamablehttp_client
from strands.tools.mcp.mcp_client import MCPClient

GATEWAY_URL = os.environ['GATEWAY_URL']  # Tutorial 04에서 가져옴

# Gateway에 연결(두 agent에서 동일한 Bazaar 대상 사용)
mcp_client = MCPClient(lambda: streamablehttp_client(
    GATEWAY_URL, timeout=timedelta(seconds=120),
))
mcp_client.__enter__()
bazaar_tools = mcp_client.list_tools_sync()

# http_request를 MCP 도구로 교체 - 나머지는 동일하게 유지
research_agent = Agent(
    model=model,
    tools=bazaar_tools,          # http_request 대신 MCP tool 사용
    plugins=[research_plugin],   # 동일한 plugin, session, wallet
    system_prompt='You are a research specialist. Use search_resources to find '
        'paid data tools on Base Sepolia, then call them with proxy_tool_call.',
)

discovery_agent = Agent(
    model=model,
    tools=bazaar_tools,          # 동일한 MCP tool, 서로 다른 plugin/budget
    plugins=[discovery_plugin],  # 서로 다른 session, wallet
    system_prompt='You are a discovery specialist. Use search_resources to find '
        'the cheapest tools under $0.10 on Base Sepolia.',
)

# Orchestrator는 변경 없이 계속 agent.as_tool() 사용
# mcp_client.__exit__(None, None, None)  # 완료 후 닫기
```

두 agent는 동일한 Gateway connection을 공유하지만 독립적인 budget 및 wallet이 있는 별도 payment plugin을 사용합니다. Gateway는 Bazaar로 routing하고 agent는 MCP를 통해 tool을 검색하며 payment plugin은 402를 처리합니다. Tutorial 04와 동일한 flow에 multi-agent orchestration을 추가한 구조입니다.